In [1]:
!pip install -U bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but

In [2]:
!pip install -q evaluate rouge_score bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer,  DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset
import torch
import numpy as np
import json

In [4]:
# Definizione delle costanti
N_EPOCHS = 3
LR = 1e-4


MAX_CONTEXT_LENGTH = 2048
MAX_CLAIM_LENGTH = 512
MAX_POST_LENGTH = MAX_CONTEXT_LENGTH - MAX_CLAIM_LENGTH

In [5]:
# Carichiamo il dataset da Hugging Face
dataset = load_dataset("DanRus21/gym_3claim")["train"]

dataset_claims.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/409 [00:00<?, ? examples/s]

In [6]:
# Splittiamo il dataset in train, validation e test
train_test = dataset.train_test_split(test_size=0.2, seed=1) # 80% train set, 20% test set

# A partire dal training set ricaviamo il validation set
train_val = train_test['train'].train_test_split(test_size=0.2, seed=1) # 80% nuovo train set, 20% validation set

train_test['train'] = train_val['train']
train_test['validation'] = train_val['test']

dataset = train_test

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'claims'],
        num_rows: 261
    })
    test: Dataset({
        features: ['text', 'claims'],
        num_rows: 82
    })
    validation: Dataset({
        features: ['text', 'claims'],
        num_rows: 66
    })
})


In [7]:
print(dataset["train"][0])

{'text': "--- POST EVENTS ---\n\nTitolo: Open Day Federazione Italiana Bodybuilding - Roma, Gennaio 2027\n\nLa FBBI (Federazione Bodybuilding e Fitness Italia) organizza il suo Open Day annuale il 10 gennaio 2027 presso il Centro Fitness Roma Eur, aperto a tutti gli atleti interessati a iniziare il percorso agonistico nel bodybuilding natural.\n\nCOSA TROVERAI\n    1. Consulenza con i Giudici: Sessioni di feedback gratuito con i giudici federali su posing, condizionamento fisico e categorie di appartenenza in base al tuo fisico attuale.\n    2. Presentazione del Calendario Gare 2027: Illustrazione del circuito nazionale e regionale di tutte le categorie, dalle prime gare amatoriali al circuito Pro.\n    3. Workshop sul Posing: 2 ore di lezione pratica di posing maschile e femminile, diviso per categorie (Bikini, Classic Physique, Men's Physique, Figure).\n\nINFORMAZIONI\n    1. Ingresso gratuito previa registrazione sul portale FBBI entro il 5 gennaio 2027.\n    2. Luogo: Centro Fitnes

In [7]:
# Carichiamo il modello
model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint, dtype=torch.float16, quantization_config=quant_config, device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
base_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

Per il Data Collator:
* model=base_model: aiuta il collator a capire i dettagli sui token speciali
* padding=True: padding dinamico alla lunghezza massima del batch corrente
* return_tensors="pt": restituisce i tensori PyTorch

In [8]:
# Carichiamo il tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, padding_side="right")

# Impostiamo il token EOS come token di padding se quest'ultimo è None
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Carichiamo il Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, padding=True, return_tensors="pt")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
# Configurazione dei parametri di LoRA
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [12]:
# Preparazione del modello quantizzato al training a k-bit
base_model = prepare_model_for_kbit_training(base_model)

model_lora = get_peft_model(base_model, lora_config)

print(model_lora)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Li

In [13]:
model_lora.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [9]:
# Formattiamo il dataset nel tipo che si aspetta SFTTrainer
def preprocess_dataset(example):
    return {
        "messages": [
            {
                "role": "system", 
                 "content": "Sei un sistema di estrazione dati in lingua italiana. Rispondi SEMPRE E SOLO in formato JSON: {'claims': ['claim1', 'claim2', 'claim3']}. Non inventare informazioni."
            },
            {
                "role": "user", 
                "content": f"Estrai le 3 affermazioni più importanti e verificabili dal post:\n\n{example['text']}"
            },
            {
                "role": "assistant", 
                "content": json.dumps({"claims": example["claims"]}, ensure_ascii=False)
            }
        ]
    }

In [10]:
# Applichiamo il processamento a tutto il dataset
dataset = dataset.map(preprocess_dataset, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/261 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

In [16]:
print(dataset["train"]["messages"][0])

[{'role': 'system', 'content': "Sei un sistema di estrazione dati in lingua italiana. Rispondi SEMPRE E SOLO in formato JSON: {'claims': ['claim1', 'claim2', 'claim3']}. Non inventare informazioni."}, {'role': 'user', 'content': "Estrai le 3 affermazioni più importanti e verificabili dal post:\n\n--- POST EVENTS ---\n\nTitolo: Open Day Federazione Italiana Bodybuilding - Roma, Gennaio 2027\n\nLa FBBI (Federazione Bodybuilding e Fitness Italia) organizza il suo Open Day annuale il 10 gennaio 2027 presso il Centro Fitness Roma Eur, aperto a tutti gli atleti interessati a iniziare il percorso agonistico nel bodybuilding natural.\n\nCOSA TROVERAI\n    1. Consulenza con i Giudici: Sessioni di feedback gratuito con i giudici federali su posing, condizionamento fisico e categorie di appartenenza in base al tuo fisico attuale.\n    2. Presentazione del Calendario Gare 2027: Illustrazione del circuito nazionale e regionale di tutte le categorie, dalle prime gare amatoriali al circuito Pro.\n   

In [11]:
import json
import re

def extract_claims(text):
    try:
        text = text.replace("```json", "").replace("```", "").strip()
        
        # Isoliamo solo la parte tra {}
        match = re.search(r'\{.*\}', text, re.DOTALL)
        
        if match:
            json_str = match.group(0)
            
            # Correggiamo il bug ]]
            json_str = json_str.replace("]]}", "]}") 
            
            obj = json.loads(json_str)
            claims = obj.get("claims", [])
            
            claims_list = []
            if isinstance(claims, str):
                claims_list = [c.strip() for c in claims.split("|") if c.strip()]
            elif isinstance(claims, list):
                claims_list = [str(c).strip() for c in claims if str(c).strip()]

            if claims_list:
                return " ".join(claims_list[:3])
            
        return "."
    except Exception as e:
        return "."

* output_dir = cartella nella quale vengono salvati i risultati del training
* per_device_train_batch_size = numero di dati elaborati contemporaneamente durante il training
* gradient_accumulation_steps = numero di step dopo il quale si aggiornano i pesi
* report_to="none" = disabilitiamo invio dei log a piattaforme esterne
* eval_strategy="epoch" = la validation loss viene calcolata alla fine di ogni epoca
* save_strategy="epoch" = il chechpoint viene salvato alla fine di ogni epoca
* max_length = lunghezza max input
* assistant_only_loss=True = i pesi vengono aggiornati solo sulle risposte dell'assistente non sul prompt dell'utente
* optim, gradient_checkpointing, gradient_checkpointing_kwargs = tecniche di ottimizzazione della memoria
* fp16 e b16 = False = modello usa la precisione standard a 32-bit

In [14]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./finetuning_output",
    learning_rate=LR,
    num_train_epochs=N_EPOCHS,
    
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    report_to="none",
    max_length=MAX_CONTEXT_LENGTH,
    assistant_only_loss=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=False,
)

trainer_lora = SFTTrainer(
    model=model_lora,                    
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

/tmp/ipykernel_58/2302637616.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


NameError: name 'model_lora' is not defined

In [20]:
trainer_lora.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.741861,0.700515,0.716460,0.840776,208141.000000
2,0.665696,0.675880,0.683732,0.844298,416282.000000
3,0.654619,0.671739,0.671020,0.845605,624423.000000


TrainOutput(global_step=99, training_loss=0.7102785206804372, metrics={'train_runtime': 1020.6563, 'train_samples_per_second': 0.767, 'train_steps_per_second': 0.097, 'total_flos': 5178291768622080.0, 'train_loss': 0.7102785206804372, 'epoch': 3.0})

In [12]:
import evaluate

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [13]:
import torch
import numpy as np
from tqdm.auto import tqdm

def evaluate_metrics(model, dataset):
    model.eval()
    training_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    
    all_preds = []
    all_labels = []
    i = 1
    
    with torch.no_grad():
        for example in tqdm(dataset, desc="Generazione claims"):
            messages = example["messages"]
            user_message = messages[1]["content"]
            label_text = messages[2]["content"]

            # Formattiamo il prompt
            prompt_str = tokenizer.apply_chat_template([
                {"role": "system", "content": "Sei un sistema di estrazione dati che lavora ESCLUSIVAMENTE sul testo fornito dall'utente. Il tuo compito è analizzare il testo e estrarre le informazioni presenti. Rispondi ESCLUSIVAMENTE in lingua italiana. Se un'informazione non è nel testo, non includerla. Rispondi SOLO in formato JSON: {'claims': ['claim1', 'claim2', 'claim3']} senza aggiungere altre informazioni."},
                {"role": "user", "content": f"Estrai le 3 affermazioni più importanti e verificabili dal post:\n\n{user_message}"}
            ], tokenize=False, add_generation_prompt=True)

            # Trasformiamo il prompt_str in token
            inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

            # Usiamo il modello in inferenza
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_CLAIM_LENGTH,        
                do_sample=False, # output deterministico
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

            # Numero di token 
            input_length = inputs.input_ids.shape[1]

            # Prendiamo solo i token generati
            generated_tokens = outputs[0][input_length:]

            # Convertiamo i token in una stringa
            pred_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

            if i == 2:
                print("-" * 50)
                print(f"POST ORIGINALE:\n{user_message}...") 
                print(f"TARGET CLAIM:\n{label_text}")
                print(f"CLAIM ESTRATTI (Modello):\n{pred_text}")
            
            all_preds.append(extract_claims(pred_text))
            all_labels.append(extract_claims(label_text))

            i += 1
    
    # Ripristina il padding originale
    tokenizer.padding_side = training_padding_side
    
    print("Calcolo ROUGE e BERTScore in corso...")
    rouge_result = rouge.compute(predictions=all_preds, references=all_labels)
    bert_result = bertscore.compute(predictions=all_preds, references=all_labels, lang="it")
    bert_f1 = np.mean(bert_result["f1"])
    
    print("\n**** METRICHE ****")
    print(f"ROUGE-1: {rouge_result['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_result['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_result['rougeL']:.4f}")
    print(f"BERTScore F1: {bert_f1:.4f}\n")

In [23]:
evaluate_metrics(model=model_lora, dataset=dataset["validation"])

Generazione claims:   0%|          | 0/66 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--------------------------------------------------
POST ORIGINALE:
Estrai le 3 affermazioni più importanti e verificabili dal post:

--- POST HOW_TO ---

Titolo: Impostare una Scheda di Forza con Periodizzazione Lineare Semplice per Principianti

La periodizzazione lineare semplice (SLP) è il metodo di programmazione più efficace e validato per principianti e intermedi. Consiste nell'aumentare il carico di allenamento in modo progressivo e sistematico, sessione dopo sessione.

1. PRINCIPIO DI BASE
    1. Sovraccarico progressivo: Ogni sessione di allenamento deve essere leggermente più sfidante della precedente. Per i principianti, aumenti di 2,5 kg a sessione sugli esercizi di spinta (panca, overhead press) e di 5 kg sugli esercizi di tiro e gambe (squat, stacco) sono sostenibili per mesi.
    2. Frequenza: 3 sessioni per settimana (es. lunedì/mercoledì/venerdì), alternando due schemi giornalieri (Giorno A e Giorno B).

2. STRUTTURA DEI GIORNI
    1. Giorno A: Squat 3x5 / Panca Piana 

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



**** METRICHE ****
ROUGE-1: 0.4314
ROUGE-2: 0.2478
ROUGE-L: 0.3391
BERTScore F1: 0.7742



In [24]:
model_lora.save_pretrained("./lora_conf_base")

In [16]:
# Ricarichiamo il modello base
model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint, dtype=torch.float16, quantization_config=quant_config, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adesso aumentiamo il LR a 2e-4

In [18]:
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

base_model = prepare_model_for_kbit_training(base_model)

model_lora = get_peft_model(base_model, lora_config)

print(model_lora.print_trainable_parameters())

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
None


In [20]:
LR=2e-4

In [21]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./finetuning_output",
    learning_rate=LR,
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=N_EPOCHS,
    report_to="none",
    max_length=MAX_CONTEXT_LENGTH, 
    assistant_only_loss=True,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer_lora = SFTTrainer(
    model=model_lora,                    
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

/tmp/ipykernel_58/2138342511.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset:   0%|          | 0/261 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/66 [00:00<?, ? examples/s]

In [22]:
trainer_lora.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.709215,0.676970,0.684034,0.846810,208141.000000
2,0.619955,0.652566,0.635080,0.849975,416282.000000
3,0.591562,0.650223,0.617021,0.851509,624423.000000


TrainOutput(global_step=99, training_loss=0.6636362123971034, metrics={'train_runtime': 996.7251, 'train_samples_per_second': 0.786, 'train_steps_per_second': 0.099, 'total_flos': 5178291768622080.0, 'train_loss': 0.6636362123971034, 'epoch': 3.0})

In [23]:
evaluate_metrics(model=model_lora, dataset=dataset["validation"])

Generazione claims:   0%|          | 0/66 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--------------------------------------------------
POST ORIGINALE:
Estrai le 3 affermazioni più importanti e verificabili dal post:

--- POST HOW_TO ---

Titolo: Impostare una Scheda di Forza con Periodizzazione Lineare Semplice per Principianti

La periodizzazione lineare semplice (SLP) è il metodo di programmazione più efficace e validato per principianti e intermedi. Consiste nell'aumentare il carico di allenamento in modo progressivo e sistematico, sessione dopo sessione.

1. PRINCIPIO DI BASE
    1. Sovraccarico progressivo: Ogni sessione di allenamento deve essere leggermente più sfidante della precedente. Per i principianti, aumenti di 2,5 kg a sessione sugli esercizi di spinta (panca, overhead press) e di 5 kg sugli esercizi di tiro e gambe (squat, stacco) sono sostenibili per mesi.
    2. Frequenza: 3 sessioni per settimana (es. lunedì/mercoledì/venerdì), alternando due schemi giornalieri (Giorno A e Giorno B).

2. STRUTTURA DEI GIORNI
    1. Giorno A: Squat 3x5 / Panca Piana 

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



**** METRICHE ****
ROUGE-1: 0.5129
ROUGE-2: 0.3159
ROUGE-L: 0.4247
BERTScore F1: 0.8078



In [24]:
model_lora.save_pretrained("./lora_base_lr2")
tokenizer.save_pretrained("./lora_base_lr2") 

('./lora_base_lr2/tokenizer_config.json',
 './lora_base_lr2/chat_template.jinja',
 './lora_base_lr2/tokenizer.json')

In [25]:
import shutil
from IPython.display import FileLink

shutil.make_archive("finetuning_dropout01_lr2", "zip", "/kaggle/working")

FileLink(r"finetuning_dropout01_lr2.zip")

/kaggle/working/finetuning_dropout01_lr2.zip

Adesso aumentiamo il numero di epoche implementando l'early stopping

In [16]:
# Ricarichiamo il modello base
model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint, dtype=torch.float16, quantization_config=quant_config, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [18]:
N_EPOCHS = 5
LR = 2e-4

In [20]:
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

base_model = prepare_model_for_kbit_training(base_model)

model_lora = get_peft_model(base_model, lora_config)

print(model_lora.print_trainable_parameters())

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
None


In [21]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./finetuning_output",
    learning_rate=LR,
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=N_EPOCHS,
    report_to="none",
    max_length=MAX_CONTEXT_LENGTH, 
    assistant_only_loss=True,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=False,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,     
    metric_for_best_model="eval_loss",
    greater_is_better=False,         
)

trainer_lora = SFTTrainer(
    model=model_lora,                    
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

/tmp/ipykernel_58/1077446501.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset:   0%|          | 0/261 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/66 [00:00<?, ? examples/s]

In [22]:
trainer_lora.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.706886,0.675628,0.691503,0.846146,208141.000000
2,0.611342,0.646094,0.610930,0.851079,416282.000000
3,0.564273,0.642640,0.585850,0.853933,624423.000000
4,0.551953,0.644626,0.571228,0.853567,832564.000000
5,0.514909,0.649658,0.528549,0.853787,1040705.000000


TrainOutput(global_step=165, training_loss=0.5971359686418013, metrics={'train_runtime': 1658.2383, 'train_samples_per_second': 0.787, 'train_steps_per_second': 0.1, 'total_flos': 8624986823577600.0, 'train_loss': 0.5971359686418013, 'epoch': 5.0})

Sopra vediamo che a partire dall'epoca 4 la validation loss aumenta, questo è un segnale di overfitting.

Adesso consideriamo la configurazione in cui aumentiamo la capacità di LoRA portando il rango da 8 a 16

In [14]:
# Ricarichiamo il modello base
model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint, dtype=torch.float16, quantization_config=quant_config, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [15]:
# Aumentiamo il rango di LoRA da 8 a 16
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

base_model = prepare_model_for_kbit_training(base_model)

model_lora = get_peft_model(base_model, lora_config)

print(model_lora.print_trainable_parameters())

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
None


In [16]:
N_EPOCHS = 3
LR = 2e-4

In [17]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./finetuning_output",
    learning_rate=LR,
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=N_EPOCHS,
    report_to="none",
    max_length=MAX_CONTEXT_LENGTH, 
    assistant_only_loss=True,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer_lora = SFTTrainer(
    model=model_lora,                    
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

/tmp/ipykernel_58/2138342511.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset:   0%|          | 0/261 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/66 [00:00<?, ? examples/s]

In [18]:
trainer_lora.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.694067,0.664553,0.665193,0.849515,208141.000000
2,0.589738,0.641845,0.597821,0.851575,416282.000000
3,0.550366,0.640720,0.585219,0.852826,624423.000000


TrainOutput(global_step=99, training_loss=0.6338240209251943, metrics={'train_runtime': 986.9304, 'train_samples_per_second': 0.793, 'train_steps_per_second': 0.1, 'total_flos': 5186888874458112.0, 'train_loss': 0.6338240209251943, 'epoch': 3.0})

In [21]:
evaluate_metrics(model=model_lora, dataset=dataset["validation"])

Generazione claims:   0%|          | 0/66 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--------------------------------------------------
POST ORIGINALE:
Estrai le 3 affermazioni più importanti e verificabili dal post:

--- POST HOW_TO ---

Titolo: Impostare una Scheda di Forza con Periodizzazione Lineare Semplice per Principianti

La periodizzazione lineare semplice (SLP) è il metodo di programmazione più efficace e validato per principianti e intermedi. Consiste nell'aumentare il carico di allenamento in modo progressivo e sistematico, sessione dopo sessione.

1. PRINCIPIO DI BASE
    1. Sovraccarico progressivo: Ogni sessione di allenamento deve essere leggermente più sfidante della precedente. Per i principianti, aumenti di 2,5 kg a sessione sugli esercizi di spinta (panca, overhead press) e di 5 kg sugli esercizi di tiro e gambe (squat, stacco) sono sostenibili per mesi.
    2. Frequenza: 3 sessioni per settimana (es. lunedì/mercoledì/venerdì), alternando due schemi giornalieri (Giorno A e Giorno B).

2. STRUTTURA DEI GIORNI
    1. Giorno A: Squat 3x5 / Panca Piana 

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



**** METRICHE ****
ROUGE-1: 0.5220
ROUGE-2: 0.3199
ROUGE-L: 0.4263
BERTScore F1: 0.8135



In [20]:
full_finetuned_model = model_lora.merge_and_unload()
full_finetuned_model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finetuned_model/tokenizer_config.json',
 './finetuned_model/chat_template.jinja',
 './finetuned_model/tokenizer.json')

In [21]:
import shutil
from IPython.display import FileLink

shutil.make_archive("finetuned_model", "zip", "/kaggle/working")

FileLink(r"finetuned_model.zip")

/kaggle/working/finetuned_model.zip

Quest'ultima configurazione è la migliore sia in termini di loss che in termini delle metriche ROUGE e BERTScore. Procediamo con la valutazione sul test set

In [24]:
evaluate_metrics(model=model_lora, dataset=dataset["test"])

Generazione claims:   0%|          | 0/82 [00:00<?, ?it/s]

--------------------------------------------------
POST ORIGINALE:
Estrai le 3 affermazioni più importanti e verificabili dal post:

--- POST NEWS ---

Titolo: Studio Pubblicato su Amino Acids Dimostra l'Efficacia della Citrullina Malato sulla Riduzione della Fatica Muscolare Acuta

Una ricerca scientifica in doppio cieco ha analizzato gli effetti dell'integrazione acuta di L-citrullina legata all'acido malico sulla clearance dell'ammoniaca e sulla risintesi del fosfocreatina durante protocolli di forza.

SCOPERTE CHIAVE DELLO STUDIO CLINICO
    1 Tampone dell'Ammoniaca via Ciclo dell'Urea: La L-citrullina stimola la clearance plasmatica dell'ammoniaca e del lattato prodotti durante la glicolisi anaerobica, ritardando l'insorgenza della fatica muscolare periferica.
    2 Incremento dell'ATP e della Risintesi di CP: Il legame con il malato (un intermedio del ciclo di Krebs) stimola la produzione di ATP ossidativo durante il riposo tra le serie, incrementando del 34% il tasso di risintes

In [25]:
# Ricarichiamo il modello base
model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint, dtype=torch.float16, quantization_config=quant_config, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [27]:
evaluate_metrics(model=base_model, dataset=dataset["test"])

Generazione claims:   0%|          | 0/82 [00:00<?, ?it/s]

--------------------------------------------------
POST ORIGINALE:
Estrai le 3 affermazioni più importanti e verificabili dal post:

--- POST NEWS ---

Titolo: Studio Pubblicato su Amino Acids Dimostra l'Efficacia della Citrullina Malato sulla Riduzione della Fatica Muscolare Acuta

Una ricerca scientifica in doppio cieco ha analizzato gli effetti dell'integrazione acuta di L-citrullina legata all'acido malico sulla clearance dell'ammoniaca e sulla risintesi del fosfocreatina durante protocolli di forza.

SCOPERTE CHIAVE DELLO STUDIO CLINICO
    1 Tampone dell'Ammoniaca via Ciclo dell'Urea: La L-citrullina stimola la clearance plasmatica dell'ammoniaca e del lattato prodotti durante la glicolisi anaerobica, ritardando l'insorgenza della fatica muscolare periferica.
    2 Incremento dell'ATP e della Risintesi di CP: Il legame con il malato (un intermedio del ciclo di Krebs) stimola la produzione di ATP ossidativo durante il riposo tra le serie, incrementando del 34% il tasso di risintes

Adesso proviamo ad estrarre i claims da un post reale generato dal nostro agente usando sia il nostro modello finetunato che la baseline

In [28]:
def generate_claims(model, tokenizer, post):
    model.eval()
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    
    with torch.no_grad():
        prompt_str = tokenizer.apply_chat_template([
            {"role": "system", "content": "Sei un sistema di estrazione dati che lavora ESCLUSIVAMENTE sul testo fornito dall'utente. Il tuo compito è analizzare il testo e estrarre le informazioni presenti. Rispondi ESCLUSIVAMENTE in lingua italiana. Se un'informazione non è nel testo, non includerla. Rispondi SOLO in formato JSON: {'claims': ['claim1', 'claim2', 'claim3']} senza aggiungere altre informazioni."},
            {"role": "user", "content": f"Estrai le 3 affermazioni più importanti e verificabili dal post:\n\n{post}"}
        ], tokenize=False, add_generation_prompt=True)
        
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=512, 
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        pred_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        print(f"CLAIM ESTRATTI:\n{pred_text}")
        
        tokenizer.padding_side = original_padding_side

In [29]:
post = """Titolo: Recensione Nike Metcon 7: La scarpa definitiva per squat e allenamento funzionale?


Categoria: REVIEW


Le Nike Metcon 7 si presentano come un punto di riferimento per chi cerca stabilità e versatilità nell'allenamento, in particolare per squat, sollevamento pesi e sessioni di CrossFit. Analizziamo le caratteristiche che le rendono una scelta popolare tra gli atleti.
CARATTERISTICHE PRINCIPALI


Stabilità per il sollevamento pesi: Suola piatta e ampia con piastra interna nel tallone per una base stabile in squat e stacchi.
Ammortizzazione React Foam: Schiuma Nike React nell'avampiede per comfort in cardio e HIIT, mantenendo stabilità.
Sistema di allacciatura e vestibilità: Chiusura a strappo per calzata sicura. Calzata stretta, considerare mezzo numero in più.
Dettagli funzionali: Rope Guard per aderenza in salita corda e clip sul tallone per verticali. Drop tacco-punta di 4 mm.

PRO


Eccellente stabilità per squat e sollevamento pesi.
Buona ammortizzazione per brevi sessioni cardio e HIIT.
Design accattivante e ampia scelta di colorazioni.
Costruzione robusta e durevole.

CONTRO


Prezzo elevato (circa $130).
Non ideali per la corsa su lunghe distanze.
La calzata può risultare stretta per alcuni.

Le Nike Metcon 7 si confermano una scelta eccellente per gli atleti che praticano sollevamento pesi, CrossFit e allenamento funzionale, offrendo un equilibrio notevole tra stabilità e comfort. Sebbene il prezzo sia un fattore da considerare e non siano le scarpe ideali per la corsa prolungata, la loro versatilità e le caratteristiche specifiche per l'allenamento le rendono un investimento valido per migliorare le prestazioni in palestra.


Fonti: ['https://www.garagegymreviews.com/nike-metcon-7-review', 'https://fitatmidlife.com/nike-metcon-7-crossfit-shoe-review', 'https://www.menshealth.com/uk/gym-wear/a37178090/nike-metcon-7-review']
"""

In [30]:
generate_claims(model_lora, tokenizer, post)

CLAIM ESTRATTI:
{"claims": ["Le Nike Metcon 7 sono dotate di schiuma Nike React nell'avampiede per ammortizzare i movimenti e mantenere la stabilità durante sessioni di cardio e HIIT.", "Il design della Nike Metcon 7 offre una chiusura a strappo per garantire una calzata sicura e una calzata stretta richiede il mezzo numero in più.", "L'analisi del post conclude che le Nike Metcon 7 sono un buon investimento per atleti che pratichiscono sollevamento pesi, CrossFit e allenamento funzionale."]}


In [33]:
generate_claims(base_model, tokenizer, post)

CLAIM ESTRATTI:
{"claims": ["Nike Metcon 7 is a point of reference for those who seek stability and versatility in training, especially for squats and weightlifting", "The Nike Metcon 7 offers excellent stability for squatting and lifting weights", "The Nike Metcon 7 provides good shock absorption for short cardio sessions and HIIT"]}
